# 03 - Visualización y comparación de resultados

Comparamos Euler y RK4 partiendo del mismo estado inicial: curvas de energía superpuestas y resumen de la solución obtenida por cada método.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.instances import FIVE_CITIES, distance_matrix_for
from src.model import HopfieldParameters, random_initial_potentials
from src.solvers import euler_integrate, rk4_integrate
from src.brute_force import brute_force_optimal
from src.analysis import (extract_route, route_as_cycle_string,
                          route_length, validate)
from src.visualization import plot_energy_comparison

In [ ]:
instance = FIVE_CITIES
d = distance_matrix_for(instance)
params = HopfieldParameters(dt=1e-5, steps=40_000)
u0 = random_initial_potentials(instance.n, params.u0, rng=np.random.default_rng(7))

res_euler = euler_integrate(u0.copy(), d, params, record_every=100)
res_rk4 = rk4_integrate(u0.copy(), d, params, record_every=100)

opt = brute_force_optimal(d)
print('Óptimo:', route_as_cycle_string(opt.route, instance), '| L =', round(opt.length, 3))
for name, res in [('Euler', res_euler), ('RK4', res_rk4)]:
    route = extract_route(res.V_final)
    print(f'{name:>6}: {route_as_cycle_string(route, instance)} '
          f'| L={route_length(route, d):.3f} '
          f'| válida={validate(res.V_final).is_permutation} '
          f'| E_final={res.final_energy:.3f}')

In [ ]:
plot_energy_comparison({
    'Euler': (res_euler.times, res_euler.energies),
    'RK4': (res_rk4.times, res_rk4.energies),
}, title='Euler vs RK4 - Energía (5 ciudades)')
plt.show()